# Stage 1 – Dataset construction (Base & Full)
Reproducing **Ni et al. 2024, *Harnessing Earnings Reports for Stock Predictions: A QLoRA-Enhanced LLM Approach*** (arXiv:2408.06634).

Runs on a normal Colab CPU runtime (no GPU needed). Data is saved to Google Drive so nothing is lost if Colab disconnects – every step can be re-run and resumes from its cache.

**Before running:** upload the `stage1_dataset` folder to `MyDrive/LLM_Project/` and put your e-mail in cell 2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/LLM_Project/stage1_dataset
!ls

In [ ]:
import os
# SEC requires a contact e-mail in the User-Agent - use YOUR name + e-mail
os.environ['SEC_USER_AGENT'] = 'Your Name your_email@example.com'
# where data is written (on Drive)
os.environ['STAGE1_DATA_DIR'] = '/content/drive/MyDrive/LLM_Project/data_smoke'
!pip install -q -U -r requirements.txt

## 1) Offline self-test (≈10 s) – must print `ALL ... CHECKS PASSED`

In [ ]:
!python tests/test_offline.py | tail -3

## 2) Smoke test on 5 companies (≈5–10 min, mostly the one-time 1.8 GB transcript download)
Check the logs and open `data_smoke/final/sample_prompts.txt` before the full run.

In [ ]:
!python step01_companies_and_transcripts.py --tickers AAPL,MSFT,JPM,NVDA,KO
!python step02_financial_growth.py
!python step03_eps_and_analyst.py
!python step04_market_and_labels.py
!python step05_build_base_full.py

In [ ]:
import pandas as pd
d = os.environ['STAGE1_DATA_DIR'] + '/final'
print(open(d + '/sample_prompts.txt').read()[:4000])
pd.read_csv(d + '/merged_events.csv').head(10)

## 3) Full run – all S&P 500 companies (≈1–2 h; safe to re-run if it stops, downloads are cached)

In [ ]:
os.environ['STAGE1_DATA_DIR'] = '/content/drive/MyDrive/LLM_Project/data'
!python step01_companies_and_transcripts.py
!python step02_financial_growth.py
!python step03_eps_and_analyst.py
!python step04_market_and_labels.py
!python step05_build_base_full.py

In [ ]:
import json
d = os.environ['STAGE1_DATA_DIR'] + '/final'
print(json.dumps(json.load(open(d + '/stats.json')), indent=2))